In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# 数据缩放示例
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
# 求解后，影子价格需要反缩放
# shadow_prices_original = shadow_prices_scaled / scaler.scale_

In [ ]:
"""
基于SBM对偶问题的影子价格计算模块。

本模块提供基于SBM模型对偶问题估计投入、产出和非期望产出
影子价格的函数。求解器首选 mosek（速度快、稳定性高）。
"""
from pyomo.environ import (
    ConcreteModel, Set, Var, Objective, Constraint, Reals,
    maximize, minimize, SolverFactory, value
)
import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings("ignore")

# 读取数据
ex4 = pd.read_stata(r"../../data/Ex4.dta")
data = ex4[["K", "L", "Y", "CO2"]]  # 提取投入、期望产出和非期望产出变量

# 设定变量个数
K = 2  # 投入变量个数
L = 1  # 期望产出变量个数
M = 1  # 非期望产出变量个数

# 准备参照集数据
xref = data.iloc[:, 0:K]
yref = data.iloc[:, K:K+L]
bref = data.iloc[:, K+L:K+L+M]

# 准备被评价DMU数据（以第1个DMU为例）
x = data.iloc[0, 0:K]
y = data.iloc[0, K:K+L]
b = data.iloc[0, K+L:K+L+M]

# 构建Pyomo模型
model = ConcreteModel()
model.I = Set(initialize=range(xref.shape[0]))  # 参照决策单元集合
model.K = Set(initialize=range(len(x)))          # 投入变量集合
model.L = Set(initialize=range(len(y)))          # 期望产出变量集合
model.M = Set(initialize=range(len(b)))          # 非期望产出变量集合

# 定义影子价格变量（施加非负约束以确保经济学含义）
model.px = Var(model.K, bounds=(0, None), within=Reals, doc='投入的影子价格')
model.py = Var(model.L, bounds=(0, None), within=Reals, doc='期望产出的影子价格')
model.pb = Var(model.M, bounds=(0, None), within=Reals, doc='非期望产出的影子价格')
model.pomega = Var(bounds=(0, None), within=Reals, doc='SBM效率的倒数')

# 目标函数：最大化 pomega
def objective_rule(model):
    """返回目标函数表达式"""
    return model.pomega * 1

# 约束1：对偶约束（参照集加权影子价格之和非正）
def first_rule(model, i):
    """第i个参照DMU的对偶约束"""
    return (
        -sum(model.px[k] * xref.iloc[i, k] for k in model.K)
        + sum(model.py[l] * yref.iloc[i, l] for l in model.L)
        - sum(model.pb[m] * bref.iloc[i, m] for m in model.M)
        <= 0
    )

# 约束2：投入影子价格的下界约束
def second_rule(model, k):
    """第k个投入变量的影子价格下界"""
    return -model.px[k] <= -1 / (len(model.K) * x[k])

# 约束3：期望产出影子价格的上界约束
def third_rule(model, l):
    """第l个期望产出变量的影子价格上界"""
    return model.pomega / (len(model.L) + len(model.M)) / y[l] - model.py[l] <= 0

# 约束4：非期望产出影子价格的上界约束
def forth_rule(model, m):
    """第m个非期望产出变量的影子价格上界"""
    return model.pomega / (len(model.L) + len(model.M)) / b[m] - model.pb[m] <= 0

# 约束5：Charnes-Cooper变换的归一化约束
def fifth_rule(model):
    """归一化约束"""
    return (
        model.pomega
        + sum(model.px[k] * x[k] for k in model.K)
        - sum(model.py[l] * y[l] for l in model.L)
        + sum(model.pb[m] * b[m] for m in model.M)
        <= 1
    )

# 组装模型
model.obj = Objective(rule=objective_rule, sense=maximize, doc='目标函数')
model.first = Constraint(model.I, rule=first_rule, doc='对偶约束')
model.second = Constraint(model.K, rule=second_rule, doc='投入下界约束')
model.third = Constraint(model.L, rule=third_rule, doc='期望产出上界约束')
model.forth = Constraint(model.M, rule=forth_rule, doc='非期望产出上界约束')
model.fifth = Constraint(rule=fifth_rule, doc='归一化约束')

# 求解
opt = SolverFactory('mosek')  # 指定 mosek 作为求解器
solution = opt.solve(model)   # 调用求解器求解

# 提取结果
px = np.asarray(list(model.px[:].value))
py = np.asarray(list(model.py[:].value))
pb = np.asarray(list(model.pb[:].value))
obj = value(model.obj)

print(f"投入影子价格 (px): {px}")
print(f"期望产出影子价格 (py): {py}")
print(f"非期望产出影子价格 (pb): {pb}")
print(f"最优目标值: {obj}")

In [ ]:
px, py, pb, obj = {}, {}, {}, {}

for j in range(data.shape[0]):
    x = data.iloc[j, 0:K]
    y = data.iloc[j, K:K+L]
    b = data.iloc[j, K+L:K+L+M]

    model = ConcreteModel()
    model.I = Set(initialize=range(xref.shape[0]))
    model.K = Set(initialize=range(len(x)))
    model.L = Set(initialize=range(len(y)))
    model.M = Set(initialize=range(len(b)))

    model.px = Var(model.K, bounds=(0, None), within=Reals, doc='投入的影子价格')
    model.py = Var(model.L, bounds=(0, None), within=Reals, doc='期望产出的影子价格')
    model.pb = Var(model.M, bounds=(0, None), within=Reals, doc='非期望产出的影子价格')
    model.pomega = Var(bounds=(0, None), within=Reals, doc='SBM效率的倒数')

    def objective_rule(model):
        return model.pomega * 1

    def first_rule(model, i):
        return (
            -sum(model.px[k] * xref.iloc[i, k] for k in model.K)
            + sum(model.py[l] * yref.iloc[i, l] for l in model.L)
            - sum(model.pb[m] * bref.iloc[i, m] for m in model.M)
            <= 0
        )

    def second_rule(model, k):
        return -model.px[k] <= -1 / (len(model.K) * x[k])

    def third_rule(model, l):
        return model.pomega / (len(model.L) + len(model.M)) / y[l] - model.py[l] <= 0

    def forth_rule(model, m):
        return model.pomega / (len(model.L) + len(model.M)) / b[m] - model.pb[m] <= 0

    def fifth_rule(model):
        return (
            model.pomega
            + sum(model.px[k] * x[k] for k in model.K)
            - sum(model.py[l] * y[l] for l in model.L)
            + sum(model.pb[m] * b[m] for m in model.M)
            <= 1
        )

    model.obj = Objective(rule=objective_rule, sense=maximize, doc='目标函数')
    model.first = Constraint(model.I, rule=first_rule, doc='对偶约束')
    model.second = Constraint(model.K, rule=second_rule, doc='投入下界约束')
    model.third = Constraint(model.L, rule=third_rule, doc='期望产出上界约束')
    model.forth = Constraint(model.M, rule=forth_rule, doc='非期望产出上界约束')
    model.fifth = Constraint(rule=fifth_rule, doc='归一化约束')

    opt = SolverFactory('mosek')
    solution = opt.solve(model)

    # 终止条件判断：optimal（最优）、infeasible（不可行）、unbounded（无界）
    if solution.solver.termination_condition == "optimal":
        px[j] = np.asarray(list(model.px[:].value))
        py[j] = np.asarray(list(model.py[:].value))
        pb[j] = np.asarray(list(model.pb[:].value))
        obj[j] = value(model.obj)

In [ ]:
from pyomo.environ import (
    ConcreteModel, Set, Var, Objective, Constraint, Reals,
    maximize, SolverFactory, value
)
import pandas as pd
import numpy as np
import re


def sbm_dual(dataframe, varname, num_inputs, num_outputs):
    """
    基于SBM模型对偶问题估计影子价格。

    求解SBM模型的对偶问题（乘数形式），获得各决策单元的投入、
    期望产出和非期望产出的影子价格。

    Parameters
    ----------
    dataframe : pd.DataFrame
        待评价决策单元的投入产出数据框，变量按投入在前、
        期望产出在中、非期望产出在后的顺序排列。
    varname : list[str]
        投入变量、期望产出变量和非期望产出变量的名称列表，
        如 ["K", "L", "Y", "CO2"]。
    num_inputs : int
        投入变量的个数（P）。
    num_outputs : int
        期望产出变量的个数（Q）。

    Returns
    -------
    pd.DataFrame
        包含各DMU的影子价格估计结果的数据框。列包括：
        - 'obj': SBM效率值的倒数
        - 'shadow price <varname>': 各变量的影子价格

    Notes
    -----
    - 本函数对影子价格施加非负约束，以确保经济学含义的合理性。
    - 求解器首选 mosek，备选 glpk 或 ipopt。
    - 关于SBM模型的原始形式，参见第6章（公式6.x）。

    Examples
    --------
    >>> ex4 = pd.read_stata("Ex4.dta")
    >>> result = sbm_dual(ex4, ["K", "L", "Y", "CO2"], 2, 1)
    >>> print(result.head())
    """
    data = dataframe.loc[:, varname]
    num_inputs_ = num_inputs
    num_outputs_ = num_outputs
    num_bad = data.shape[1] - num_inputs_ - num_outputs_

    xref = data.iloc[:, 0:num_inputs_]
    yref = data.iloc[:, num_inputs_:num_inputs_+num_outputs_]
    bref = data.iloc[:, num_inputs_+num_outputs_:num_inputs_+num_outputs_+num_bad]

    xcol = varname[0:num_inputs_]
    ycol = varname[num_inputs_:num_inputs_+num_outputs_]
    bcol = varname[num_inputs_+num_outputs_:num_inputs_+num_outputs_+num_bad]

    px, py, pb, obj = {}, {}, {}, {}

    for j in range(data.shape[0]):
        x = data.iloc[j, 0:num_inputs_]
        y = data.iloc[j, num_inputs_:num_inputs_+num_outputs_]
        b = data.iloc[j, num_inputs_+num_outputs_:num_inputs_+num_outputs_+num_bad]

        model = ConcreteModel()
        model.I = Set(initialize=range(xref.shape[0]))
        model.K = Set(initialize=range(len(x)))
        model.L = Set(initialize=range(len(y)))
        model.M = Set(initialize=range(len(b)))

        model.px = Var(model.K, bounds=(0, None), within=Reals, doc='投入的影子价格')
        model.py = Var(model.L, bounds=(0, None), within=Reals, doc='期望产出的影子价格')
        model.pb = Var(model.M, bounds=(0, None), within=Reals, doc='非期望产出的影子价格')
        model.pomega = Var(bounds=(0, None), within=Reals, doc='SBM效率的倒数')

        def objective_rule(model):
            return model.pomega * 1

        def first_rule(model, i):
            return (
                -sum(model.px[k] * xref.iloc[i, k] for k in model.K)
                + sum(model.py[l] * yref.iloc[i, l] for l in model.L)
                - sum(model.pb[m] * bref.iloc[i, m] for m in model.M)
                <= 0
            )

        def second_rule(model, k):
            return -model.px[k] <= -1 / (len(model.K) * x[k])

        def third_rule(model, l):
            return model.pomega / (len(model.L) + len(model.M)) / y[l] - model.py[l] <= 0

        def forth_rule(model, m):
            return model.pomega / (len(model.L) + len(model.M)) / b[m] - model.pb[m] <= 0

        def fifth_rule(model):
            return (
                model.pomega
                + sum(model.px[k] * x[k] for k in model.K)
                - sum(model.py[l] * y[l] for l in model.L)
                + sum(model.pb[m] * b[m] for m in model.M)
                <= 1
            )

        model.obj = Objective(rule=objective_rule, sense=maximize, doc='目标函数')
        model.first = Constraint(model.I, rule=first_rule, doc='对偶约束')
        model.second = Constraint(model.K, rule=second_rule, doc='投入下界约束')
        model.third = Constraint(model.L, rule=third_rule, doc='期望产出上界约束')
        model.forth = Constraint(model.M, rule=forth_rule, doc='非期望产出上界约束')
        model.fifth = Constraint(rule=fifth_rule, doc='归一化约束')

        opt = SolverFactory('mosek')
        solution = opt.solve(model)

        if solution.solver.termination_condition == "optimal":
            px[j] = np.asarray(list(model.px[:].value))
            py[j] = np.asarray(list(model.py[:].value))
            pb[j] = np.asarray(list(model.pb[:].value))
            obj[j] = value(model.obj)

    objdf = pd.DataFrame(obj, index=["obj"]).T
    pxdf = pd.DataFrame(px, index=xcol).T
    pxdf.columns = pxdf.columns.map(lambda x: "shadow price " + str(x))
    pydf = pd.DataFrame(py, index=ycol).T
    pydf.columns = pydf.columns.map(lambda y: "shadow price " + str(y))
    pbdf = pd.DataFrame(pb, index=bcol).T
    pbdf.columns = pbdf.columns.map(lambda b: "shadow price " + str(b))

    pdf = pd.concat([pxdf, pydf], axis=1)
    pdf = pd.concat([pdf, pbdf], axis=1)
    redata = pd.concat([objdf, pdf], axis=1)

    return redata

In [ ]:
import pandas as pd

ex4 = pd.read_stata(r"../../data/Ex4.dta")
result = sbm_dual(ex4, ["K", "L", "Y", "CO2"], 2, 1)
print(result.head())

In [ ]:
def sbm_dual_select(dataframe, varname, num_inputs, num_outputs,
                    eval_query, ref_query):
    """
    基于SBM对偶问题估计影子价格（支持评价单元和参照集筛选）。

    Parameters
    ----------
    dataframe : pd.DataFrame
        待评价决策单元的投入产出数据框。
    varname : list[str]
        投入、期望产出和非期望产出变量的名称列表，
        如 ["K", "L", "Y", "CO2"]。
    num_inputs : int
        投入变量的个数。
    num_outputs : int
        期望产出变量的个数。
    eval_query : str
        传入数据框.query()方法的参数，用于筛选待评价DMU，
        如 "dmu==1"、"dmu==[1,2,3]"。
    ref_query : str
        传入数据框.query()方法的参数，用于筛选参照集DMU，
        如 "dmu==1"、"dmu==[1,2,3]"。

    Returns
    -------
    pd.DataFrame
        包含各DMU影子价格估计结果的数据框。
    """
    index_list = dataframe.query(eval_query).index
    index_list_ref = dataframe.query(ref_query).index

    data = dataframe.loc[index_list, varname]
    dataref = dataframe.loc[index_list_ref, varname]

    num_inputs_ = num_inputs
    num_outputs_ = num_outputs
    num_bad = data.shape[1] - num_inputs_ - num_outputs_

    xref = dataref.iloc[:, 0:num_inputs_]
    yref = dataref.iloc[:, num_inputs_:num_inputs_+num_outputs_]
    bref = dataref.iloc[:, num_inputs_+num_outputs_:num_inputs_+num_outputs_+num_bad]

    xcol = varname[0:num_inputs_]
    ycol = varname[num_inputs_:num_inputs_+num_outputs_]
    bcol = varname[num_inputs_+num_outputs_:num_inputs_+num_outputs_+num_bad]

    px, py, pb, obj = {}, {}, {}, {}

    for j in data.index:
        x = data.loc[j, xcol]
        y = data.loc[j, ycol]
        b = data.loc[j, bcol]

        model = ConcreteModel()
        model.I = Set(initialize=dataref.index)
        model.K = Set(initialize=range(len(x)))
        model.L = Set(initialize=range(len(y)))
        model.M = Set(initialize=range(len(b)))

        model.px = Var(model.K, bounds=(0, None), within=Reals, doc='投入的影子价格')
        model.py = Var(model.L, bounds=(0, None), within=Reals, doc='期望产出的影子价格')
        model.pb = Var(model.M, bounds=(0, None), within=Reals, doc='非期望产出的影子价格')
        model.pomega = Var(bounds=(0, None), within=Reals, doc='SBM效率的倒数')

        def objective_rule(model):
            return model.pomega * 1

        def first_rule(model, i):
            return (
                -sum(model.px[k] * xref.iloc[i, k] for k in model.K)
                + sum(model.py[l] * yref.iloc[i, l] for l in model.L)
                - sum(model.pb[m] * bref.iloc[i, m] for m in model.M)
                <= 0
            )

        def second_rule(model, k):
            return -model.px[k] <= -1 / (len(model.K) * x[k])

        def third_rule(model, l):
            return model.pomega / (len(model.L) + len(model.M)) / y[l] - model.py[l] <= 0

        def forth_rule(model, m):
            return model.pomega / (len(model.L) + len(model.M)) / b[m] - model.pb[m] <= 0

        def fifth_rule(model):
            return (
                model.pomega
                + sum(model.px[k] * x[k] for k in model.K)
                - sum(model.py[l] * y[l] for l in model.L)
                + sum(model.pb[m] * b[m] for m in model.M)
                <= 1
            )

        model.obj = Objective(rule=objective_rule, sense=maximize, doc='目标函数')
        model.first = Constraint(model.I, rule=first_rule, doc='对偶约束')
        model.second = Constraint(model.K, rule=second_rule, doc='投入下界约束')
        model.third = Constraint(model.L, rule=third_rule, doc='期望产出上界约束')
        model.forth = Constraint(model.M, rule=forth_rule, doc='非期望产出上界约束')
        model.fifth = Constraint(rule=fifth_rule, doc='归一化约束')

        opt = SolverFactory('mosek')
        solution = opt.solve(model)

        if solution.solver.termination_condition == "optimal":
            px[j] = np.asarray(list(model.px[:].value))
            py[j] = np.asarray(list(model.py[:].value))
            pb[j] = np.asarray(list(model.pb[:].value))
            obj[j] = value(model.obj)

    objdf = pd.DataFrame(obj, index=["obj"]).T
    pxdf = pd.DataFrame(px, index=xcol).T
    pxdf.columns = pxdf.columns.map(lambda x: "shadow price " + str(x))
    pydf = pd.DataFrame(py, index=ycol).T
    pydf.columns = pydf.columns.map(lambda y: "shadow price " + str(y))
    pbdf = pd.DataFrame(pb, index=bcol).T
    pbdf.columns = pbdf.columns.map(lambda b: "shadow price " + str(b))

    pdf = pd.concat([pxdf, pydf], axis=1)
    pdf = pd.concat([pdf, pbdf], axis=1)
    redata = pd.concat([objdf, pdf], axis=1)

    return redata

In [ ]:
def sbm_dual_formula(formula, dataframe, eval_query=None, ref_query=None):
    """
    基于SBM对偶问题估计影子价格（支持公式字符串调用）。

    Parameters
    ----------
    formula : str
        公式字符串，格式为 "Y : CO2 = K L"。
        等号左边为产出变量（冒号前为期望产出，冒号后为非期望产出），
        等号右边为投入变量。
    dataframe : pd.DataFrame
        待评价决策单元的投入产出数据框。
    eval_query : str or None, default None
        传入数据框.query()方法的参数，用于筛选待评价DMU。
        若为None，则评价所有DMU。
    ref_query : str or None, default None
        传入数据框.query()方法的参数，用于筛选参照集DMU。
        若为None，则使用全部DMU作为参照集。

    Returns
    -------
    pd.DataFrame
        包含各DMU影子价格估计结果的数据框。
    """
    px, py, pb, obj = {}, {}, {}, {}

    if eval_query is None:
        index_list = dataframe.index
    else:
        index_list = dataframe.query(eval_query).index

    if ref_query is None:
        index_list_ref = dataframe.index
    else:
        index_list_ref = dataframe.query(ref_query).index

    # 解析公式字符串
    input_vars = formula.split('=')[1].strip(' ')
    xcol = re.compile(' +').sub(' ', input_vars).split(' ')
    output_vars = formula.split('=')[0].split(':')[0].strip(' ')
    ycol = re.compile(' +').sub(' ', output_vars).split(' ')
    unoutput_vars = formula.split('=')[0].split(':')[1].strip(' ')
    bcol = re.compile(' +').sub(' ', unoutput_vars).split(' ')

    data = dataframe.loc[index_list, xcol + ycol + bcol]
    dataref = dataframe.loc[index_list_ref, xcol + ycol + bcol]

    xref = dataref.loc[:, xcol]
    yref = dataref.loc[:, ycol]
    bref = dataref.loc[:, bcol]

    for j in data.index:
        x = data.loc[j, xcol]
        y = data.loc[j, ycol]
        b = data.loc[j, bcol]

        model = ConcreteModel()
        model.I = Set(initialize=dataref.index)
        model.K = Set(initialize=range(len(x)))
        model.L = Set(initialize=range(len(y)))
        model.M = Set(initialize=range(len(b)))

        model.px = Var(model.K, bounds=(0, None), within=Reals, doc='投入的影子价格')
        model.py = Var(model.L, bounds=(0, None), within=Reals, doc='期望产出的影子价格')
        model.pb = Var(model.M, bounds=(0, None), within=Reals, doc='非期望产出的影子价格')
        model.pomega = Var(bounds=(0, None), within=Reals, doc='SBM效率的倒数')

        def objective_rule(model):
            return model.pomega * 1

        def first_rule(model, i):
            return (
                -sum(model.px[k] * xref.iloc[i, k] for k in model.K)
                + sum(model.py[l] * yref.iloc[i, l] for l in model.L)
                - sum(model.pb[m] * bref.iloc[i, m] for m in model.M)
                <= 0
            )

        def second_rule(model, k):
            return -model.px[k] <= -1 / (len(model.K) * x[k])

        def third_rule(model, l):
            return model.pomega / (len(model.L) + len(model.M)) / y[l] - model.py[l] <= 0

        def forth_rule(model, m):
            return model.pomega / (len(model.L) + len(model.M)) / b[m] - model.pb[m] <= 0

        def fifth_rule(model):
            return (
                model.pomega
                + sum(model.px[k] * x[k] for k in model.K)
                - sum(model.py[l] * y[l] for l in model.L)
                + sum(model.pb[m] * b[m] for m in model.M)
                <= 1
            )

        model.obj = Objective(rule=objective_rule, sense=maximize, doc='目标函数')
        model.first = Constraint(model.I, rule=first_rule, doc='对偶约束')
        model.second = Constraint(model.K, rule=second_rule, doc='投入下界约束')
        model.third = Constraint(model.L, rule=third_rule, doc='期望产出上界约束')
        model.forth = Constraint(model.M, rule=forth_rule, doc='非期望产出上界约束')
        model.fifth = Constraint(rule=fifth_rule, doc='归一化约束')

        opt = SolverFactory('mosek')
        solution = opt.solve(model)

        if solution.solver.termination_condition == "optimal":
            px[j] = np.asarray(list(model.px[:].value))
            py[j] = np.asarray(list(model.py[:].value))
            pb[j] = np.asarray(list(model.pb[:].value))
            obj[j] = value(model.obj)

    objdf = pd.DataFrame(obj, index=["obj"]).T
    pxdf = pd.DataFrame(px, index=xcol).T
    pxdf.columns = pxdf.columns.map(lambda x: "shadow price " + str(x))
    pydf = pd.DataFrame(py, index=ycol).T
    pydf.columns = pydf.columns.map(lambda y: "shadow price " + str(y))
    pbdf = pd.DataFrame(pb, index=bcol).T
    pbdf.columns = pbdf.columns.map(lambda b: "shadow price " + str(b))

    pdf = pd.concat([pxdf, pydf], axis=1)
    pdf = pd.concat([pdf, pbdf], axis=1)
    redata = pd.concat([objdf, pdf], axis=1)

    return redata

In [ ]:
import pandas as pd

ex4 = pd.read_stata(r"../../data/Ex4.dta")
result = sbm_dual_formula("Y : CO2 = K L", ex4, "t==[1,2]", "t==[1,2,3]")
print(result)

In [ ]:
import pandas as pd
import numpy as np

# 构造一个退化数据集：3个DMU，1投入1产出1非期望产出
data_degen = pd.DataFrame({
    'dmu': ['A', 'B', 'C'],
    'x': [1.0, 2.0, 2.0],   # 投入
    'y': [1.0, 2.0, 2.0],   # 期望产出
    'b': [1.0, 1.0, 1.0]    # 非期望产出
})

# DMU B和C的投入产出完全相同，导致前沿退化
# 运行sbm_dual后，可观察到影子价格的不稳定性
result = sbm_dual(data_degen, ['x', 'y', 'b'], 1, 1)
print(result)

In [ ]:
from pyomo.environ import (
    ConcreteModel, Set, Var, Objective, Constraint, Reals,
    minimize, SolverFactory, value
)
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# 读取数据
ex5 = pd.read_stata(r"../../data/Ex5.dta")
data = ex5[['labor', 'capital', 'energy', 'gdp', 'co2']]

# 设定变量个数
K = 3  # 投入变量个数（labor, capital, energy）
L = 1  # 期望产出变量个数（gdp）
M = 1  # 非期望产出变量个数（co2）

# 准备参照集数据
xref = data.iloc[:, 0:K]
yref = data.iloc[:, K:K+L]
bref = data.iloc[:, K+L:K+L+M]

# 准备被评价DMU数据（以第1个DMU为例）
x = data.iloc[0, 0:K]
y = data.iloc[0, K:K+L]
b = data.iloc[0, K+L:K+L+M]

# 变量名
xcol = xref.columns
ycol = yref.columns
bcol = bref.columns

# 设定方向向量（默认：投入减少、产出增加、非期望产出减少）
gx = [-1] * len(xcol)
gy = [1] * len(ycol)
gb = [-1] * len(bcol)

# 计算权重（默认等权重）
weight = []
denominator = int(gx[0] != 0) + int(gy[0] != 0) + int(gb[0] != 0)
for _ in range(len(xcol)):
    weight.append(1.0 / denominator / len(xcol))
for _ in range(len(ycol)):
    weight.append(1.0 / denominator / len(ycol))
for _ in range(len(bcol)):
    weight.append(1.0 / denominator / len(bcol))

iweight = weight[0:len(xcol)]
oweight = weight[len(xcol):len(xcol)+len(ycol)]
bweight = weight[len(xcol)+len(ycol):len(xcol)+len(ycol)+len(bcol)]

# 构建Pyomo模型
model = ConcreteModel()
model.I = Set(initialize=range(xref.shape[0]))  # 参照决策单元集合
model.K = Set(initialize=range(len(xcol)))       # 投入变量集合
model.L = Set(initialize=range(len(ycol)))       # 期望产出变量集合
model.M = Set(initialize=range(len(bcol)))       # 非期望产出变量集合

# 定义影子价格变量（施加非负约束）
model.px = Var(model.K, bounds=(0, None), within=Reals, doc='投入的影子价格')
model.py = Var(model.L, bounds=(0, None), within=Reals, doc='期望产出的影子价格')
model.pb = Var(model.M, bounds=(0, None), within=Reals, doc='非期望产出的影子价格')
model.pomega = Var(bounds=(0, None), within=Reals, doc='方向距离函数值')

# 目标函数：最小化虚拟利润
def objective_rule(model):
    """返回目标函数表达式（虚拟利润）"""
    return (
        sum(model.px[k] * x[k] for k in model.K)
        - sum(model.py[l] * y[l] for l in model.L)
        + sum(model.pb[j] * b[j] for j in model.M)
        + model.pomega * 1
    )

# 约束1：对偶约束（参照集加权影子价格之和非负）
def first_rule(model, i):
    """第i个参照DMU的对偶约束"""
    return (
        sum(model.px[k] * xref.loc[i, xcol[k]] for k in model.K)
        - sum(model.py[l] * yref.loc[i, ycol[l]] for l in model.L)
        + sum(model.pb[m] * bref.loc[i, bcol[m]] for m in model.M)
        + model.pomega * 1
        >= 0
    )

# 约束2：投入影子价格的下界约束
def second_rule(model, k):
    """第k个投入变量的影子价格下界"""
    return -gx[k] * x[k] * model.px[k] >= iweight[k]

# 约束3：期望产出影子价格的下界约束
def third_rule(model, l):
    """第l个期望产出变量的影子价格下界"""
    return gy[l] * y[l] * model.py[l] >= oweight[l]

# 约束4：非期望产出影子价格的下界约束
def forth_rule(model, j):
    """第j个非期望产出变量的影子价格下界"""
    return -gb[j] * b[j] * model.pb[j] >= bweight[j]

# 组装模型
model.obj = Objective(rule=objective_rule, sense=minimize, doc='目标函数')
model.first = Constraint(model.I, rule=first_rule, doc='对偶约束')
model.second = Constraint(model.K, rule=second_rule, doc='投入下界约束')
model.third = Constraint(model.L, rule=third_rule, doc='期望产出下界约束')
model.forth = Constraint(model.M, rule=forth_rule, doc='非期望产出下界约束')

# 求解
opt = SolverFactory('mosek')
solution = opt.solve(model)

# 提取结果
px = np.asarray(list(model.px[:].value))
py = np.asarray(list(model.py[:].value))
pb = np.asarray(list(model.pb[:].value))
obj = value(model.obj)

print(f"投入影子价格 (px): {px}")
print(f"期望产出影子价格 (py): {py}")
print(f"非期望产出影子价格 (pb): {pb}")
print(f"最优目标值: {obj}")

In [ ]:
px, py, pb, obj = {}, {}, {}, {}

for j in range(data.shape[0]):
    x = data.iloc[j, 0:K]
    y = data.iloc[j, K:K+L]
    b = data.iloc[j, K+L:K+L+M]

    model = ConcreteModel()
    model.I = Set(initialize=range(xref.shape[0]))
    model.K = Set(initialize=range(len(xcol)))
    model.L = Set(initialize=range(len(ycol)))
    model.M = Set(initialize=range(len(bcol)))

    model.px = Var(model.K, bounds=(0, None), within=Reals, doc='投入的影子价格')
    model.py = Var(model.L, bounds=(0, None), within=Reals, doc='期望产出的影子价格')
    model.pb = Var(model.M, bounds=(0, None), within=Reals, doc='非期望产出的影子价格')
    model.pomega = Var(bounds=(0, None), within=Reals, doc='方向距离函数值')

    def objective_rule(model):
        return (
            sum(model.px[k] * x[k] for k in model.K)
            - sum(model.py[l] * y[l] for l in model.L)
            + sum(model.pb[jj] * b[jj] for jj in model.M)
            + model.pomega * 1
        )

    def first_rule(model, i):
        return (
            sum(model.px[k] * xref.loc[i, xcol[k]] for k in model.K)
            - sum(model.py[l] * yref.loc[i, ycol[l]] for l in model.L)
            + sum(model.pb[m] * bref.loc[i, bcol[m]] for m in model.M)
            + model.pomega * 1
            >= 0
        )

    def second_rule(model, k):
        return -gx[k] * x[k] * model.px[k] >= iweight[k]

    def third_rule(model, l):
        return gy[l] * y[l] * model.py[l] >= oweight[l]

    def forth_rule(model, jj):
        return -gb[jj] * b[jj] * model.pb[jj] >= bweight[jj]

    model.obj = Objective(rule=objective_rule, sense=minimize, doc='目标函数')
    model.first = Constraint(model.I, rule=first_rule, doc='对偶约束')
    model.second = Constraint(model.K, rule=second_rule, doc='投入下界约束')
    model.third = Constraint(model.L, rule=third_rule, doc='期望产出下界约束')
    model.forth = Constraint(model.M, rule=forth_rule, doc='非期望产出下界约束')

    opt = SolverFactory('mosek')
    solution = opt.solve(model)

    if solution.solver.termination_condition == "optimal":
        px[j] = np.asarray(list(model.px[:].value))
        py[j] = np.asarray(list(model.py[:].value))
        pb[j] = np.asarray(list(model.pb[:].value))
        obj[j] = value(model.obj)

In [ ]:
def nddf_dual(dataframe, varname, num_inputs, num_outputs):
    """
    基于NDDF模型对偶问题估计影子价格。

    求解NDDF模型的对偶问题（乘数形式），获得各决策单元的投入、
    期望产出和非期望产出的影子价格。

    Parameters
    ----------
    dataframe : pd.DataFrame
        待评价决策单元的投入产出数据框，变量按投入在前、
        期望产出在中、非期望产出在后的顺序排列。
    varname : list[str]
        投入变量、期望产出变量和非期望产出变量的名称列表，
        如 ["labor", "capital", "energy", "gdp", "co2"]。
    num_inputs : int
        投入变量的个数。
    num_outputs : int
        期望产出变量的个数。

    Returns
    -------
    pd.DataFrame
        包含各DMU的影子价格估计结果的数据框。

    Notes
    -----
    - 默认方向向量：投入减少(-1)、期望产出增加(+1)、非期望产出减少(-1)。
    - 默认权重：等权重分配。
    - 如需自定义方向向量和权重，请使用 nddf_dual_advanced 函数。
    """
    data = dataframe.loc[:, varname]
    num_inputs_ = num_inputs
    num_outputs_ = num_outputs
    num_bad = data.shape[1] - num_inputs_ - num_outputs_

    xref = data.iloc[:, 0:num_inputs_]
    yref = data.iloc[:, num_inputs_:num_inputs_+num_outputs_]
    bref = data.iloc[:, num_inputs_+num_outputs_:num_inputs_+num_outputs_+num_bad]

    xcol = xref.columns
    ycol = yref.columns
    bcol = bref.columns

    # 默认方向向量
    gx = [-1] * len(xcol)
    gy = [1] * len(ycol)
    gb = [-1] * len(bcol)

    # 默认权重
    weight = []
    denominator = int(gx[0] != 0) + int(gy[0] != 0) + int(gb[0] != 0)
    for _ in range(len(xcol)):
        weight.append(1.0 / denominator / len(xcol))
    for _ in range(len(ycol)):
        weight.append(1.0 / denominator / len(ycol))
    for _ in range(len(bcol)):
        weight.append(1.0 / denominator / len(bcol))

    iweight = weight[0:len(xcol)]
    oweight = weight[len(xcol):len(xcol)+len(ycol)]
    bweight = weight[len(xcol)+len(ycol):len(xcol)+len(ycol)+len(bcol)]

    px, py, pb, obj = {}, {}, {}, {}

    for j in range(data.shape[0]):
        x = data.iloc[j, 0:num_inputs_]
        y = data.iloc[j, num_inputs_:num_inputs_+num_outputs_]
        b = data.iloc[j, num_inputs_+num_outputs_:num_inputs_+num_outputs_+num_bad]

        model = ConcreteModel()
        model.I = Set(initialize=range(xref.shape[0]))
        model.K = Set(initialize=range(len(xcol)))
        model.L = Set(initialize=range(len(ycol)))
        model.M = Set(initialize=range(len(bcol)))

        model.px = Var(model.K, bounds=(0, None), within=Reals, doc='投入的影子价格')
        model.py = Var(model.L, bounds=(0, None), within=Reals, doc='期望产出的影子价格')
        model.pb = Var(model.M, bounds=(0, None), within=Reals, doc='非期望产出的影子价格')
        model.pomega = Var(bounds=(0, None), within=Reals, doc='方向距离函数值')

        def objective_rule(model):
            return (
                sum(model.px[k] * x[k] for k in model.K)
                - sum(model.py[l] * y[l] for l in model.L)
                + sum(model.pb[jj] * b[jj] for jj in model.M)
                + model.pomega * 1
            )

        def first_rule(model, i):
            return (
                sum(model.px[k] * xref.loc[i, xcol[k]] for k in model.K)
                - sum(model.py[l] * yref.loc[i, ycol[l]] for l in model.L)
                + sum(model.pb[m] * bref.loc[i, bcol[m]] for m in model.M)
                + model.pomega * 1
                >= 0
            )

        def second_rule(model, k):
            return -gx[k] * x[k] * model.px[k] >= iweight[k]

        def third_rule(model, l):
            return gy[l] * y[l] * model.py[l] >= oweight[l]

        def forth_rule(model, jj):
            return -gb[jj] * b[jj] * model.pb[jj] >= bweight[jj]

        model.obj = Objective(rule=objective_rule, sense=minimize, doc='目标函数')
        model.first = Constraint(model.I, rule=first_rule, doc='对偶约束')
        model.second = Constraint(model.K, rule=second_rule, doc='投入下界约束')
        model.third = Constraint(model.L, rule=third_rule, doc='期望产出下界约束')
        model.forth = Constraint(model.M, rule=forth_rule, doc='非期望产出下界约束')

        opt = SolverFactory('mosek')
        solution = opt.solve(model)

        if solution.solver.termination_condition == "optimal":
            px[j] = np.asarray(list(model.px[:].value))
            py[j] = np.asarray(list(model.py[:].value))
            pb[j] = np.asarray(list(model.pb[:].value))
            obj[j] = value(model.obj)

    objdf = pd.DataFrame(obj, index=["obj"]).T
    pxdf = pd.DataFrame(px, index=xcol).T
    pxdf.columns = pxdf.columns.map(lambda x: "shadow price " + str(x))
    pydf = pd.DataFrame(py, index=ycol).T
    pydf.columns = pydf.columns.map(lambda y: "shadow price " + str(y))
    pbdf = pd.DataFrame(pb, index=bcol).T
    pbdf.columns = pbdf.columns.map(lambda b: "shadow price " + str(b))

    pdf = pd.concat([pxdf, pydf], axis=1)
    pdf = pd.concat([pdf, pbdf], axis=1)
    redata = pd.concat([objdf, pdf], axis=1)

    return redata

In [ ]:
def nddf_dual_advanced(formula, dataframe, gx=None, gy=None, gb=None,
                       weight=None, eval_query=None, ref_query=None):
    """
    基于NDDF对偶问题估计影子价格（高级版本）。

    支持自定义方向向量、权重，以及评价单元和参照集的筛选。

    Parameters
    ----------
    formula : str
        公式字符串，格式为 "期望产出:非期望产出=投入1 投入2 ..."，
        如 "gdp : co2 = labor capital energy"。
    dataframe : pd.DataFrame
        待评价决策单元的投入产出数据框。
    gx : list or None, default None
        投入方向向量。默认为 [-1, -1, ...]，表示投入减少方向。
    gy : list or None, default None
        期望产出方向向量。默认为 [1, 1, ...]，表示期望产出增加方向。
    gb : list or None, default None
        非期望产出方向向量。默认为 [-1, -1, ...]，表示非期望产出减少方向。
    weight : list or None, default None
        各变量的权重向量。若为None，则使用等权重分配。
    eval_query : str or None, default None
        传入数据框.query()方法的参数，用于筛选待评价DMU。
        若为None，则评价所有DMU。
    ref_query : str or None, default None
        传入数据框.query()方法的参数，用于筛选参照集DMU。
        若为None，则使用全部DMU作为参照集。

    Returns
    -------
    pd.DataFrame
        包含各DMU影子价格估计结果的数据框。
    """
    px, py, pb, obj = {}, {}, {}, {}

    # 处理查询条件
    if eval_query is None:
        index_list = dataframe.index
    else:
        index_list = dataframe.query(eval_query).index

    if ref_query is None:
        index_list_ref = dataframe.index
    else:
        index_list_ref = dataframe.query(ref_query).index

    # 解析公式字符串
    input_vars = formula.split('=')[1].strip(' ')
    xcol = re.compile(' +').sub(' ', input_vars).split(' ')
    output_vars = formula.split('=')[0].split(':')[0].strip(' ')
    ycol = re.compile(' +').sub(' ', output_vars).split(' ')
    unoutput_vars = formula.split('=')[0].split(':')[1].strip(' ')
    bcol = re.compile(' +').sub(' ', unoutput_vars).split(' ')

    data = dataframe.loc[index_list, xcol + ycol + bcol]
    dataref = dataframe.loc[index_list_ref, xcol + ycol + bcol]

    xref = dataref.loc[:, xcol]
    yref = dataref.loc[:, ycol]
    bref = dataref.loc[:, bcol]

    # 处理方向向量（使用None判断默认值）
    if gx is None:
        gx = [-1] * len(xcol)
    if gy is None:
        gy = [1] * len(ycol)
    if gb is None:
        gb = [-1] * len(bcol)

    # 处理权重（使用None判断默认值）
    if weight is None:
        weight = []
        denominator = int(gx[0] != 0) + int(gy[0] != 0) + int(gb[0] != 0)
        for _ in range(len(xcol)):
            weight.append(1.0 / denominator / len(xcol))
        for _ in range(len(ycol)):
            weight.append(1.0 / denominator / len(ycol))
        for _ in range(len(bcol)):
            weight.append(1.0 / denominator / len(bcol))

    iweight = weight[0:len(xcol)]
    oweight = weight[len(xcol):len(xcol)+len(ycol)]
    bweight = weight[len(xcol)+len(ycol):len(xcol)+len(ycol)+len(bcol)]

    for j in data.index:
        x = data.loc[j, xcol]
        y = data.loc[j, ycol]
        b = data.loc[j, bcol]

        model = ConcreteModel()
        model.I = Set(initialize=range(xref.shape[0]))
        model.K = Set(initialize=range(len(xcol)))
        model.L = Set(initialize=range(len(ycol)))
        model.M = Set(initialize=range(len(bcol)))

        model.px = Var(model.K, bounds=(0, None), within=Reals, doc='投入的影子价格')
        model.py = Var(model.L, bounds=(0, None), within=Reals, doc='期望产出的影子价格')
        model.pb = Var(model.M, bounds=(0, None), within=Reals, doc='非期望产出的影子价格')
        model.pomega = Var(bounds=(0, None), within=Reals, doc='方向距离函数值')

        def objective_rule(model):
            return (
                sum(model.px[k] * x[k] for k in model.K)
                - sum(model.py[l] * y[l] for l in model.L)
                + sum(model.pb[jj] * b[jj] for jj in model.M)
                + model.pomega * 1
            )

        def first_rule(model, i):
            return (
                sum(model.px[k] * xref.loc[i, xcol[k]] for k in model.K)
                - sum(model.py[l] * yref.loc[i, ycol[l]] for l in model.L)
                + sum(model.pb[m] * bref.loc[i, bcol[m]] for m in model.M)
                + model.pomega * 1
                >= 0
            )

        def second_rule(model, k):
            return -gx[k] * x[k] * model.px[k] >= iweight[k]

        def third_rule(model, l):
            return gy[l] * y[l] * model.py[l] >= oweight[l]

        def forth_rule(model, jj):
            return -gb[jj] * b[jj] * model.pb[jj] >= bweight[jj]

        model.obj = Objective(rule=objective_rule, sense=minimize, doc='目标函数')
        model.first = Constraint(model.I, rule=first_rule, doc='对偶约束')
        model.second = Constraint(model.K, rule=second_rule, doc='投入下界约束')
        model.third = Constraint(model.L, rule=third_rule, doc='期望产出下界约束')
        model.forth = Constraint(model.M, rule=forth_rule, doc='非期望产出下界约束')

        opt = SolverFactory('mosek')
        solution = opt.solve(model)

        if solution.solver.termination_condition == "optimal":
            px[j] = np.asarray(list(model.px[:].value))
            py[j] = np.asarray(list(model.py[:].value))
            pb[j] = np.asarray(list(model.pb[:].value))
            obj[j] = value(model.obj)

    objdf = pd.DataFrame(obj, index=["obj"]).T
    pxdf = pd.DataFrame(px, index=xcol).T
    pxdf.columns = pxdf.columns.map(lambda x: "shadow price " + str(x))
    pydf = pd.DataFrame(py, index=ycol).T
    pydf.columns = pydf.columns.map(lambda y: "shadow price " + str(y))
    pbdf = pd.DataFrame(pb, index=bcol).T
    pbdf.columns = pbdf.columns.map(lambda b: "shadow price " + str(b))

    pdf = pd.concat([pxdf, pydf], axis=1)
    pdf = pd.concat([pdf, pbdf], axis=1)
    redata = pd.concat([objdf, pdf], axis=1)

    return redata

In [ ]:
import pandas as pd

ex5 = pd.read_stata(r"../../data/Ex5.dta")
result = nddf_dual_advanced(
    'gdp : co2 = labor capital energy',
    ex5,
    gx=[-1, -1, -1],
    gy=[1],
    gb=[-1],
    weight=None,
    eval_query=None,
    ref_query=None
)
print(result.head())

In [ ]:
import pandas as pd

# 读取省级数据
data = pd.read_stata(r"province_data.dta")

# 估计碳排放影子价格
result = nddf_dual(
    data,
    ['labor', 'capital', 'energy', 'gdp', 'co2'],
    num_inputs=3,
    num_outputs=1
)

# 提取CO2影子价格
co2_shadow_price = result['shadow price co2']

# 描述性统计
print(co2_shadow_price.describe())

# 将影子价格与碳市场价格比较
# 例如：中国全国碳市场2021年平均价格约40元/吨
carbon_market_price = 40  # 元/吨CO2
co2_shadow_price_vs_market = co2_shadow_price / carbon_market_price
print(f"影子价格/市场价格比率: {co2_shadow_price_vs_market.describe()}")

In [ ]:
import pandas as pd
import numpy as np

# 行业数据
industries = pd.DataFrame({
    'industry': ['钢铁', '电力', '化工', '建材', '造纸'],
    'so2_emission': [1000, 2000, 800, 600, 400],   # 当前SO2排放量（吨）
    'shadow_price': [500, 300, 800, 400, 600]       # SO2影子价格（元/吨）
})

total_reduction = 0.2 * industries['so2_emission'].sum()  # 总减排目标20%

# 等边际成本分配：从影子价格最低的行业开始减排
industries_sorted = industries.sort_values('shadow_price')
industries_sorted['reduction'] = np.minimum(
    industries_sorted['so2_emission'] * 0.5,  # 最多减排50%
    total_reduction / len(industries)  # 初始均分
)

# 计算总减排成本
total_cost = (industries_sorted['reduction']
              * industries_sorted['shadow_price']).sum()
print(f"总减排成本: {total_cost:.2f} 万元")

In [ ]:
import numpy as np
import pandas as pd
from pyomo.environ import SolverFactory


def bootstrap_shadow_price(dataframe, varname, num_inputs, num_outputs,
                           model_func, b=2000, alpha=0.05, seed=42):
    """
    使用Bootstrap方法估计影子价格的置信区间。

    基于Simar-Wilson（1998, 2000）的Bootstrap DEA方法，
    对影子价格进行偏差校正并获得置信区间。

    Parameters
    ----------
    dataframe : pd.DataFrame
        原始投入产出数据。
    varname : list[str]
        变量名称列表。
    num_inputs : int
        投入变量个数。
    num_outputs : int
        期望产出变量个数。
    model_func : callable
        影子价格估计函数（如 sbm_dual 或 nddf_dual）。
    b : int, default 2000
        Bootstrap重复次数。
    alpha : float, default 0.05
        显著性水平（0.05对应95%置信区间）。
    seed : int, default 42
        随机数种子，保证结果可重复。

    Returns
    -------
    pd.DataFrame
        包含点估计、偏差校正估计、置信区间上下限的数据框。
    """
    rng = np.random.default_rng(seed)
    n = dataframe.shape[0]

    # 步骤1：原始估计
    original_result = model_func(dataframe, varname, num_inputs, num_outputs)

    # 提取影子价格列（非obj列）
    sp_cols = [c for c in original_result.columns if c != 'obj']
    original_sp = original_result[sp_cols]

    # 存储Bootstrap结果
    bootstrap_results = []

    # 步骤2：Bootstrap循环
    for _ in range(b):
        # 有放回抽样
        boot_idx = rng.integers(0, n, size=n)
        boot_data = dataframe.iloc[boot_idx].reset_index(drop=True)

        try:
            boot_result = model_func(boot_data, varname, num_inputs, num_outputs)
            boot_sp = boot_result[sp_cols]
            bootstrap_results.append(boot_sp)
        except Exception:
            # 若Bootstrap样本求解失败，跳过
            continue

    # 合并Bootstrap结果
    all_boot = pd.concat(bootstrap_results, ignore_index=True)

    # 步骤3-4：计算偏差校正的置信区间
    results = {}
    for col in sp_cols:
        point_est = original_sp[col].values
        boot_est = all_boot[col].values.reshape(b, n)

        # 偏差
        bias = np.nanmean(boot_est, axis=0) - point_est

        # 偏差校正百分位置信区间
        lower = 2 * point_est - np.nanquantile(boot_est, 1 - alpha / 2, axis=0)
        upper = 2 * point_est - np.nanquantile(boot_est, alpha / 2, axis=0)

        results[f'{col}_point'] = point_est
        results[f'{col}_bias_corrected'] = point_est - bias
        results[f'{col}_lower'] = lower
        results[f'{col}_upper'] = upper

    return pd.DataFrame(results, index=original_result.index)


# 使用示例
# ci_result = bootstrap_shadow_price(
#     ex4, ["K", "L", "Y", "CO2"], 2, 1,
#     model_func=sbm_dual, b=2000
# )
# print(ci_result.head())

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# 数据标准化示例
def scale_for_shadow_price(data, input_cols, output_cols, bad_cols):
    """
    对投入产出数据进行标准化，以提高数值稳定性。

    Parameters
    ----------
    data : pd.DataFrame
        原始数据。
    input_cols, output_cols, bad_cols : list[str]
        投入、期望产出、非期望产出的列名。

    Returns
    -------
    scaled_data : pd.DataFrame
        标准化后的数据。
    scaler_params : dict
        标准化参数（用于将影子价格反变换到原始尺度）。
    """
    scaler = StandardScaler()
    all_cols = input_cols + output_cols + bad_cols
    scaled_values = scaler.fit_transform(data[all_cols])

    scaled_data = data.copy()
    scaled_data[all_cols] = scaled_values

    scaler_params = {
        'mean': scaler.mean_,
        'scale': scaler.scale_,
        'input_cols': input_cols,
        'output_cols': output_cols,
        'bad_cols': bad_cols
    }

    return scaled_data, scaler_params

In [ ]:
opt = SolverFactory('mosek')

# 设置严格容差
opt.options['intpnt_co_tol_pfeas'] = 1e-9   # 原始可行性容差
opt.options['intpnt_co_tol_dfeas'] = 1e-9   # 对偶可行性容差
opt.options['intpnt_co_tol_rel_gap'] = 1e-9  # 相对间隙容差

solution = opt.solve(model)

In [ ]:
def check_degeneracy(model, tol=1e-8):
    """
    检查DEA对偶问题是否存在退化。

    通过检查对偶约束的松弛变量是否接近零来判断退化。
    如果多个约束同时紧致，可能存在退化。

    Parameters
    ----------
    model : pyomo.environ.ConcreteModel
        已求解的Pyomo模型。
    tol : float, default 1e-8
        判断约束是否紧致的容差。

    Returns
    -------
    bool
        是否存在退化。
    """
    tight_count = 0
    for constr in model.first.values():
        slack = constr.uslack()  # 上界松弛
        if abs(slack) < tol:
            tight_count += 1

    # 如果超过维度数的约束同时紧致，可能存在退化
    is_degenerate = tight_count > len(model.K)
    return is_degenerate